In [ ]:
from typing import Literal

from dotenv import  load_dotenv
from langchain_deepseek import ChatDeepSeek
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.checkpoint.postgres import PostgresSaver
from langgraph.graph import MessagesState, StateGraph,START,END
from langchain.tools import tool
from langgraph.prebuilt import ToolNode, ToolRuntime
from langgraph.types import Command
from loguru import logger
from langchain.messages import HumanMessage,ToolMessage
from rich import print as rprint
load_dotenv(override = True)
#3. 声明状态 -> 传入的时候 直接写入HumanMessage
class OverAllState(MessagesState):
    output:str
    weather_res:str
    new_res:str
#1. 添加工具调用
@tool(parse_docstring=True)
def get_weather(city:str,runtime:ToolRuntime) ->Command:
    """
    查询指定城市的当日天气

    Args:
        city:城市名称
    """
    res=f"{city}天气晴朗，微风"
    tool_call_id=runtime.tool_call_id
    tool_msg=ToolMessage(tool_call_id=tool_call_id,content=res)
    return Command(
        update={
            "weather_res":res,
            "messages":[tool_msg]
        }
    )

@tool(parse_docstring=True)
def get_news(home_or_abroad:bool,runtime:ToolRuntime) ->Command:
    """
    查询国内外的新闻

    Args:
        home_or_abroad: 查询国内或者国外的新闻 True:国内新闻 False:国外新闻
    """
    if home_or_abroad:
        res="Kimi 新模型发布"
    else:
        res="Anthropic 暂停新模型访问"
    tool_call_id=runtime.tool_call_id
    tool_msg=ToolMessage(tool_call_id=tool_call_id,content=res)
    return Command(
        update={
            "new_res":res,
            "messages":[tool_msg]
        }
    )
model = ChatDeepSeek(
    model = "deepseek-v4-flash",
    extra_body=
    {
        "thinking":{
            "type":"disabled"
        }
    }
)

#2. 将工具绑定到模型上
tools_by_name={"get_weather":get_weather, "get_news":get_news}
tools=[get_weather,get_news]
model_with_tools=model.bind_tools(tools=tools)


#4. 定义节点
def llm_node(state: OverAllState) -> OverAllState:
    msg=state["messages"]
    #调用大模型
    res=model_with_tools.invoke(msg)
    return{
        "messages":[res],
        "output":res.content
    }


def router(state:OverAllState) -> Literal["tool_node",END]:
    if state["messages"][-1].tool_calls:
        return "tool_node"
    return END

#5. 构建图
builder = StateGraph(state_schema=OverAllState)
builder.add_node("llm_node",llm_node)
builder.add_node("tool_node",ToolNode(tools=tools))

builder.add_edge(START,"llm_node")
builder.add_conditional_edges("llm_node",router,path_map=["tool_node",END])
builder.add_edge("tool_node","llm_node")

graph = builder.compile()

from IPython.display import display
display(graph)

res = graph.invoke({"messages":[HumanMessage("今天北京的天气怎么样?国内有什么新闻?")]})
print(res)
for msg in res["messages"]:
    rprint(msg)
